In [62]:
%load_ext autoreload
%autoreload 2

import os, sys, h5py, gc, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import concurrent.futures as cf
from pathlib import Path
from torch.utils.data import DataLoader

REPO_ROOT = "../"
proj = Path(REPO_ROOT).resolve()

# allow: import ADCNN.*
sys.path.insert(0, str(proj))

# allow: import utils.*  (where utils == ADCNN/utils)
sys.path.insert(0, str(proj / "ADCNN"))

import ADCNN.inference.postprocess as postprocess
from ADCNN.data.datasets import H5TiledDataset
from ADCNN.inference.predict import load_model, predict_tiles_to_full
import ADCNN.evaluation.detection as evals
import ADCNN.evaluation.threshold_scan as threshold_scan

test_h5 = "../DATA/test.h5"
test_csv = "../DATA/test.csv"
MODEL_CKPT = "../checkpoints/ckpt_best.pt"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
ds_te = H5TiledDataset(test_h5,  tile=128, k_sigma=5.0)

test_loader = DataLoader(
    ds_te,
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    persistent_workers=False,
    prefetch_factor=None,
)

test_catalog = pd.read_csv(test_csv)
with h5py.File(test_h5, "r") as _f:
    gt_test = _f["masks"][:].astype(np.uint8)
    stack_fp = _f["real_labels"][:].astype(np.uint16)
print("Test tiles:", len(ds_te))

Test tiles: 51200


In [63]:
def get_stack_metrics (catalog, fp_mask):
    tp = (catalog["stack_detection"]==True).sum()
    fn = (catalog["stack_detection"]==False).sum()
    fp = fp_mask.max(axis=(1, 2)).sum()
    return  {"TP": tp, "FP": fp, "FN": fn, "TN": pd.NA}

print ("5 sigma threshold")
with h5py.File("../DATA/test.h5", "r") as _f:
    stack_fp = _f["real_labels"][:].astype(np.uint16)
evals.print_confusion_matrix(get_stack_metrics (pd.read_csv("../DATA/test.csv"), stack_fp), title="Object-wise Confusion Matrix")
with h5py.File("../DATA/snr4/test.h5", "r") as _f:
    stack_fp = _f["real_labels"][:].astype(np.uint16)
print ("4 sigma threshold")
evals.print_confusion_matrix(get_stack_metrics (pd.read_csv("../DATA/snr4/test.csv"), stack_fp), title="Object-wise Confusion Matrix")
with h5py.File("../DATA/snr3/test.h5", "r") as _f:
    stack_fp = _f["real_labels"][:].astype(np.uint16)
print ("3 sigma threshold")
evals.print_confusion_matrix(get_stack_metrics (pd.read_csv("../DATA/snr3/test.csv"), stack_fp), title="Object-wise Confusion Matrix")

5 sigma threshold
Object-wise Confusion Matrix
F1 Score: 0.0071, F2 Score: 0.0175
                 Predicted Negative  Predicted Positive
Actual Negative                <NA>              128764
Actual Positive                 535                 465

4 sigma threshold
Object-wise Confusion Matrix
F1 Score: 0.0069, F2 Score: 0.0169
                 Predicted Negative  Predicted Positive
Actual Negative                <NA>              161413
Actual Positive                 438                 562

3 sigma threshold
Object-wise Confusion Matrix
F1 Score: 0.0069, F2 Score: 0.0169
                 Predicted Negative  Predicted Positive
Actual Negative                <NA>              199316
Actual Positive                 309                 691

